# 1. import + mount drive + shared helpers

In [ ]:
import os
import re
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

from google.colab import drive
drive.mount('/content/drive')

# -----------------------------
# 字型（中文/韓文）
# -----------------------------
!apt-get -qq -y install fonts-wqy-zenhei fonts-nanum > /dev/null

try:
    font_path = '/usr/share/fonts/truetype/wqy/wqy-zenhei.ttc'
    fm.fontManager.addfont(font_path)
    plt.rcParams['font.family'] = fm.FontProperties(fname=font_path).get_name()
except:
    pass

plt.rcParams['axes.unicode_minus'] = False

# -----------------------------
# 共用工具
# -----------------------------
def ensure_dir(path):
    if not os.path.exists(path):
        os.makedirs(path)

def get_lang_key(lang):
    if pd.isna(lang):
        return 'en'
    lang = str(lang).lower()
    if lang.startswith('zh'):
        return 'zh'
    if lang.startswith('ko'):
        return 'ko'
    return 'en'

def pick_text_for_scan(row):
    """
    優先用 context；若沒有再退回 sentence
    """
    if 'context' in row and pd.notna(row['context']) and str(row['context']).strip():
        return str(row['context'])
    return str(row.get('sentence', ''))

def compile_keyword_pattern(keywords, lang='en'):
    """
    English 用 boundary；中文/韓文直接字串匹配
    長詞優先，避免短詞吃掉長詞
    """
    kws = sorted(set([str(k).strip() for k in keywords if str(k).strip()]), key=len, reverse=True)
    if get_lang_key(lang) == 'en':
        parts = [rf'(?<!\w){re.escape(k.lower())}(?!\w)' for k in kws]
        return re.compile('|'.join(parts), flags=re.IGNORECASE), kws
    else:
        parts = [re.escape(k) for k in kws]
        return re.compile('|'.join(parts)), kws

def find_matches(text, pattern, lang='en'):
    if pd.isna(text):
        return []
    t = str(text)
    if get_lang_key(lang) == 'en':
        return re.findall(pattern, t.lower())
    return re.findall(pattern, t)

def standardize_mentions_df(df):
    """
    將新 pipeline mentions df 做基本標準化
    """
    df = df.copy()

    # 必要欄位檢查
    required = ['state_new', 'gender_new', 'article_id', 'sentence']
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(f"mentions file 缺少必要欄位: {missing}")

    # 若沒有 context，就補成 sentence
    if 'context' not in df.columns:
        df['context'] = df['sentence']

    # 若沒有 platform / lang，嘗試補
    if 'platform' not in df.columns:
        df['platform'] = 'unknown'
    if 'lang' not in df.columns:
        df['lang'] = 'en'

    # 若沒有 sent_idx，就用 sentence 當去重輔助
    if 'sent_idx' not in df.columns:
        df['sent_idx'] = np.nan

    return df

def prepare_ghost_mentions(df):
    """
    新 pipeline 版本：
    只保留超自然 + 已知 gender 的 mentions
    並做句級去重
    """
    df = standardize_mentions_df(df)

    df_ghosts = df[
        (df['state_new'] == 'Supernatural') &
        (df['gender_new'].isin(['Female', 'Male']))
    ].copy()

    # 優先用 sent_idx 去重；否則退回 sentence
    if df_ghosts['sent_idx'].notna().any():
        df_ghosts = df_ghosts.drop_duplicates(subset=['article_id', 'sent_idx', 'gender_new']).copy()
    else:
        df_ghosts = df_ghosts.drop_duplicates(subset=['article_id', 'sentence', 'gender_new']).copy()

    return df_ghosts

def run_domain_analysis_from_all(
    input_file,
    output_dir,
    platform_name,
    groups_dict,
    output_prefix,
    scan_col_preference='context',
    norm_per=1000
):
    ensure_dir(output_dir)

    df_ent = pd.read_csv(input_file)
    df_ent = standardize_mentions_df(df_ent)

    # 只取該平台
    df_ent = df_ent[df_ent['platform'].str.lower() == platform_name.lower()].copy()

    df_ghosts = prepare_ghost_mentions(df_ent)

    total_female_base = len(df_ghosts[df_ghosts['gender_new'] == 'Female'])
    total_male_base   = len(df_ghosts[df_ghosts['gender_new'] == 'Male'])

    print(f"[{platform_name}] female base = {total_female_base}, male base = {total_male_base}")

    summary_results = []
    keyword_results = []
    all_matched_rows = []

    platform_lang = get_lang_key(df_ghosts['lang'].dropna().iloc[0]) if len(df_ghosts) else 'en'

    for category, keywords in groups_dict.items():
        pattern, sorted_keywords = compile_keyword_pattern(keywords, lang=platform_lang)

        cat_f_total = 0
        cat_m_total = 0
        word_counts = {kw: {'Female': 0, 'Male': 0} for kw in sorted_keywords}

        for _, row in df_ghosts.iterrows():
            scan_text = pick_text_for_scan(row) if scan_col_preference == 'context' else str(row['sentence'])
            gender = row['gender_new']
            found_matches = find_matches(scan_text, pattern, lang=row['lang'])

            if found_matches:
                hits = len(found_matches)

                if gender == 'Female':
                    cat_f_total += hits
                else:
                    cat_m_total += hits

                all_matched_rows.append({
                    "Platform": row.get('platform', platform_name),
                    "Category": category,
                    "article_id": row['article_id'],
                    "Story_ID": row['article_id'],
                    "Gender": gender,
                    "gender_new": gender,
                    "state_new": row['state_new'],
                    "entity_canon": row.get('entity_canon', ''),
                    "Found_Keywords": ", ".join(sorted(set(found_matches))),
                    "Sentence": row['sentence'],
                    "Context": row.get('context', row['sentence']),
                    "Hit_Count": hits
                })

                for m in found_matches:
                    for kw in sorted_keywords:
                        if get_lang_key(row['lang']) == 'en':
                            if m.lower() == kw.lower():
                                word_counts[kw][gender] += 1
                                break
                        else:
                            if m == kw:
                                word_counts[kw][gender] += 1
                                break

        total = cat_f_total + cat_m_total
        if total > 0:
            f_norm = (cat_f_total / total_female_base) * norm_per if total_female_base > 0 else 0
            m_norm = (cat_m_total / total_male_base) * norm_per if total_male_base > 0 else 0

            summary_results.append({
                "Category": category,
                "Female_Freq": cat_f_total,
                "Male_Freq": cat_m_total,
                "Total_N": total,
                "Female_Ratio": cat_f_total / total,
                "Female_Norm_Freq": f_norm,
                "Male_Norm_Freq": m_norm
            })

            for kw, counts in word_counts.items():
                if (counts['Female'] + counts['Male']) > 0:
                    keyword_results.append({
                        "Category": category,
                        "Keyword": kw,
                        "Female_Freq": counts['Female'],
                        "Male_Freq": counts['Male'],
                        "Female_Norm_Freq": (counts['Female'] / total_female_base) * norm_per if total_female_base > 0 else 0,
                        "Male_Norm_Freq": (counts['Male'] / total_male_base) * norm_per if total_male_base > 0 else 0,
                        "Total_Freq": counts['Female'] + counts['Male']
                    })

    summary_df = pd.DataFrame(summary_results)
    keyword_df = pd.DataFrame(keyword_results)
    details_df = pd.DataFrame(all_matched_rows)

    summary_path = os.path.join(output_dir, f"{platform_name}_{output_prefix}_summary.csv")
    keyword_path = os.path.join(output_dir, f"{platform_name}_{output_prefix}_keyword_level_freq.csv")
    details_path = os.path.join(output_dir, f"{platform_name}_{output_prefix}_matched_sentences_details.csv")

    summary_df.to_csv(summary_path, index=False, encoding='utf-8-sig')
    keyword_df.to_csv(keyword_path, index=False, encoding='utf-8-sig')
    details_df.to_csv(details_path, index=False, encoding='utf-8-sig')

    print("saved:", summary_path)
    print("saved:", keyword_path)
    print("saved:", details_path)


    if len(summary_df) > 0:
        plot_df = summary_df.sort_values('Female_Freq', ascending=True).copy()
        y = np.arange(len(plot_df))
        h = 0.35

        plt.figure(figsize=(10, max(4, len(plot_df) * 0.6)))
        plt.barh(y + h/2, plot_df['Female_Freq'], h, label='Female', color='#f0a1a8')
        plt.barh(y - h/2, plot_df['Male_Freq'], h, label='Male', color='#a1c4f0')
        plt.yticks(y, plot_df['Category'])
        plt.xlabel("Frequency (Number of Hits)")
        plt.title(f"{platform_name.upper()} {output_prefix.upper()} Summary (Raw Frequency)")
        plt.legend()
        plt.tight_layout()
        plt.savefig(
            os.path.join(output_dir, f"{platform_name}_{output_prefix}_summary_raw_bar.png"),
            dpi=300
    )
        plt.show()

    if len(summary_df) > 0:
        plot_df = summary_df.sort_values('Female_Norm_Freq', ascending=True).copy()
        y = np.arange(len(plot_df))
        h = 0.35

        plt.figure(figsize=(10, max(4, len(plot_df) * 0.6)))
        plt.barh(y + h/2, plot_df['Female_Norm_Freq'], h, label='Female', color='#f0a1a8')
        plt.barh(y - h/2, plot_df['Male_Norm_Freq'], h, label='Male', color='#a1c4f0')
        plt.yticks(y, plot_df['Category'])
        plt.xlabel(f"Normalized Frequency (per {norm_per} ghost mentions)")
        plt.title(f"{platform_name.upper()} {output_prefix.upper()} Summary (Normalized)")
        plt.legend()
        plt.tight_layout()
        plt.savefig(os.path.join(output_dir, f"{platform_name}_{output_prefix}_summary_norm_bar.png"), dpi=300)
        plt.show()

    if len(keyword_df) > 0:
        for cat in keyword_df['Category'].unique():
            cat_data = keyword_df[keyword_df['Category'] == cat].sort_values(by='Total_Freq', ascending=True)
            plt.figure(figsize=(10, max(3, len(cat_data) * 0.5 + 2)))
            y = np.arange(len(cat_data))
            h = 0.35

            plt.barh(y + h/2, cat_data['Female_Freq'], h, label='Female', color='#f0a1a8')
            plt.barh(y - h/2, cat_data['Male_Freq'], h, label='Male', color='#a1c4f0')
            plt.yticks(y, cat_data['Keyword'])
            plt.title(f"{platform_name.upper()} {output_prefix.upper()} Keyword Frequency: {cat}")
            plt.xlabel("Frequency (Number of Hits)")
            plt.legend()
            plt.tight_layout()
            plt.savefig(os.path.join(output_dir, f"{platform_name}_{output_prefix}_bar_chart_{cat}.png"), dpi=300)
            plt.show()

    if len(keyword_df) > 0:
      for cat in keyword_df['Category'].unique():
        cat_data = keyword_df[keyword_df['Category'] == cat].copy()
        cat_data['Total_Norm_Freq'] = cat_data['Female_Norm_Freq'] + cat_data['Male_Norm_Freq']
        cat_data = cat_data.sort_values(by='Total_Norm_Freq', ascending=True)

        plt.figure(figsize=(10, max(3, len(cat_data) * 0.5 + 2)))
        y = np.arange(len(cat_data))
        h = 0.35

        plt.barh(y + h/2, cat_data['Female_Norm_Freq'], h, label='Female', color='#f0a1a8')
        plt.barh(y - h/2, cat_data['Male_Norm_Freq'], h, label='Male', color='#a1c4f0')
        plt.yticks(y, cat_data['Keyword'])
        plt.title(f"{platform_name.upper()} {output_prefix.upper()} Keyword Frequency (Normalized): {cat}")
        plt.xlabel(f"Normalized Frequency (per {norm_per} ghost mentions)")
        plt.legend()
        plt.tight_layout()
        plt.savefig(
            os.path.join(output_dir, f"{platform_name}_{output_prefix}_norm_bar_chart_{cat}.png"),
            dpi=300
        )
        plt.show()

    return summary_df, keyword_df, details_df

# path config

In [ ]:
# -----------------------------
# 新 pipeline mentions 檔路徑
# -----------------------------
ALL_MENTIONS_PATH = "/content/drive/MyDrive/Thesis_CT_Kelly/NEW PIPELINE FILES/mentions_with_content_words.csv"

# 原始全文語料路徑（close reading 需要）
RAW_PATHS = {
    "ptt": "/content/drive/MyDrive/ghost-story-gender-age-analysis/data/raw/new_ptt_experience_articles_all.csv",
    "theqoo": "/content/drive/MyDrive/ghost-story-gender-age-analysis/data/raw/theqoo_experience_articles_v2.csv",
    "reddit": "/content/drive/MyDrive/ghost_runs_english/reddit_merged_stories.csv"
}

# 輸出資料夾
SPACE_OUTPUT_DIR = "/content/drive/MyDrive/gender_space_frequency_results_newpipeline"
COLOR_OUTPUT_DIR = "/content/drive/MyDrive/gender_color_analysis_newpipeline"
METHOD_OUTPUT_DIR = "/content/drive/MyDrive/gender_haunting_methods_analysis_newpipeline"
CLOSE_READING_DIR = "/content/drive/MyDrive/thesis_ghost_Close_Reading_Files_newpipeline"

ensure_dir(SPACE_OUTPUT_DIR)
ensure_dir(COLOR_OUTPUT_DIR)
ensure_dir(METHOD_OUTPUT_DIR)
ensure_dir(CLOSE_READING_DIR)

# place / space keyword groups

In [ ]:
SPACE_GROUPS = {
    "ptt": {
        "Indoor (Domestic)": ['房間', '房門','廚房','臥房','阿太房','空房','病房',
                    '客房','房屋','暗房','樓房','新房子','雅房','大房子','書房',
                    '小房','平房','套房','茅房','小書房','睡房','老房子','房子','大樓',
                    '閣樓','儲藏室','和室','休息室','山莊',
                    '紅眠床','大床','病床','彈簧床','床位','床沿','床邊'],
        "Sleep_Dream": ['常夢','夢見','鄰居托夢','時托夢','夢境','入夢','夢到那','夢到','噩夢',
                  '惡夢','實夢','夢中聞','夢鄉','夢話','夢樂透','做夢','作夢','托夢','夢醒',
                  '夢過','姊托夢','夢遊','夢魘','睡著','睡醒','睡覺','午睡','睡到',
                  '晚睡','熟睡','睡眠','昏睡','睡意','半睡半醒','睡死','打瞌睡','入睡'],
        "Bathroom": ['廁所'],
        "Window": ['窗戶', '窗簾','落地窗','氣窗','紙窗','窗台','玻璃窗','門窗','車窗',
                   '紗窗','小窗','外窗','櫥窗','鐵窗'],
        "Hallway": ['走廊', '黑長廊','長廊'],
        "Stairs": ['樓梯角','樓梯口','樓梯'],
        "School": ['學校', '教室', '宿舍','女生宿舍','男生宿舍','校園','夜校','校舍','住校',
                   '研究室','實驗室','大學','小學','中學','宿營','研究所'],
        "Outdoor": ['金寶山', '柏油路', '車站', '岔路','公路','小路','巡山','騎樓','佛光山','登山','金山',
                    '高山','山徑','武山','濱海','霞海','南海','海灘','斑馬線','黑森林','教會','墓地',
                    '墳墓區','墳墓','墓園','捷運站','車道','車場','公園','廟口','妻廟','祖師廟',
                    '桃園大廟','古廟','公廟','廟寺','大廟'],
        "Military Setting": ['部隊','查房','巡房','軍械室','國軍','軍區','軍卡車',
                     '美軍','軍方','軍隊','軍艦','軍機','軍工','行軍','軍校','軍旅','海軍',
                     '日軍','軍徽','單位','兵部','指揮部','旅部','營部',
                     '全營','營舍','營區','摸哨','哨站','站哨','撤哨','哨屋',
                     '接哨','上哨','下哨','外哨','哨點','哨所','內務櫃','戰車','甲車','碉堡']
    },
    "theqoo": {
        "Indoor (Domestic)": ['안방','침대','주방','방바닥','거실','독실','창고','숙소'],
        "Sleep_Dream": ['꿈꾸','꿈속','잠들','잠결','잠자','낮잠'],
        "Bathroom": ['화장실'],
        "Window": ['창문', '창가','창호지'],
        "Hallway": ['복도','현관'],
        "Stairs": ['계단'],
        "School": ['고등학교','교실','강의실','학교','초등학교','기숙사','대학'],
        "Outdoor": ['숲', '무덤','강원도','횡단보도','산길','사거리','뒷산',
                    '첩첩산중','교회','지하철','공원'],
        "Military Setting": ['중대','대대장','부대','초소','행보관']
    },
    "reddit": {
        "Indoor (Domestic)": ['room', 'bedroom', 'kitchen','house','playroom','storeroom',
                    'living room','livingroom','sunroom','guesthouse',
                    'warehouse','haunted house','townhouse','farmhouse','coachhouse',
                    'abandoned house','bedside'],
        "Sleep_Dream": ['dream','sleep paralysis','asleep','dreamed','dreaming','half-asleep',
                  'sleeper','sleepwalking','sleeping',
                  'half asleep','sleep','sleepwalk'],
        "Bathroom": ['bathroom', 'restroom', 'washroom','toilet'],
        "Window": ['window','windowsill'],
        "Hallway": ['hallway', 'corridor','hall','aisle','passage'],
        "Stairs": ['stair','stairwell','staircase','downstairs','upstairs','downstair','stairs'],
        "School": ['abandoned school','highschool','school', 'preschool',
                   'classroom', 'dormitory','dorm','elementary'],
        "Outdoor": ['church','ballroom','shophouse','lighthouse','dirtroad','roadside','backroad',
                    'railroad','crossroad','road','forest','woodland','woode','greenwood',
                    'graveyard','gravesite','cemetery','park','temple','subway','sidewalk'],
        "Military Setting": ['military', 'army','military base','soldier','navy']
    }
}

# theqoo / space

In [ ]:
summary_theqoo_space, keyword_theqoo_space, details_theqoo_space = run_domain_analysis_from_all(
    input_file=ALL_MENTIONS_PATH,
    output_dir=SPACE_OUTPUT_DIR,
    platform_name="theqoo",
    groups_dict=SPACE_GROUPS["theqoo"],
    output_prefix="space",
    scan_col_preference='context',
    norm_per=1000
)

# ptt / space

In [ ]:
summary_ptt_space, keyword_ptt_space, details_ptt_space = run_domain_analysis_from_all(
    input_file=ALL_MENTIONS_PATH,
    output_dir=SPACE_OUTPUT_DIR,
    platform_name="ptt",
    groups_dict=SPACE_GROUPS["ptt"],
    output_prefix="space",
    scan_col_preference='context',
    norm_per=1000
)

# reddit / space

In [ ]:
summary_reddit_space, keyword_reddit_space, details_reddit_space = run_domain_analysis_from_all(
    input_file=ALL_MENTIONS_PATH,
    output_dir=SPACE_OUTPUT_DIR,
    platform_name="reddit",
    groups_dict=SPACE_GROUPS["reddit"],
    output_prefix="space",
    scan_col_preference='context',
    norm_per=1000
)

# close reading pullout（space）

In [ ]:
def load_raw_articles(platform):
    df_raw = pd.read_csv(RAW_PATHS[platform]).copy()

    # 如果 raw 檔本身已有 article_id，就直接沿用
    if 'article_id' not in df_raw.columns:
        df_raw['article_id'] = df_raw.index

    # 保留舊欄位名，方便後面 close reading 相容
    df_raw['Story_ID'] = df_raw['article_id']

    return df_raw


SPACE_DETAIL_PATHS = {
    "ptt": os.path.join(SPACE_OUTPUT_DIR, "ptt_space_matched_sentences_details.csv"),
    "theqoo": os.path.join(SPACE_OUTPUT_DIR, "theqoo_space_matched_sentences_details.csv"),
    "reddit": os.path.join(SPACE_OUTPUT_DIR, "reddit_space_matched_sentences_details.csv")
}

SPACE_CLOSE_DIR = os.path.join(CLOSE_READING_DIR, "Space_Analysis")
ensure_dir(SPACE_CLOSE_DIR)


def pull_space_close_reading(platform):
    detail_path = SPACE_DETAIL_PATHS[platform]
    if not os.path.exists(detail_path):
        print("missing:", detail_path)
        return

    df_details = pd.read_csv(detail_path)
    df_raw = load_raw_articles(platform)

    merge_key = 'article_id' if 'article_id' in df_details.columns else 'Story_ID'

    # 只保留需要的全文欄位，並改名避免混淆
    raw_cols = ['article_id', 'Story_ID']
    if 'content' in df_raw.columns:
        raw_cols.append('content')
    elif 'text' in df_raw.columns:
        raw_cols.append('text')
    else:
        raise ValueError(f"No content/text column found in raw file for {platform}")

    df_raw_sub = df_raw[raw_cols].copy()
    df_raw_sub = df_raw_sub.rename(columns={
        'content': 'full_text',
        'text': 'full_text'
    })

    df_final = pd.merge(
        df_details,
        df_raw_sub,
        on=merge_key,
        how='left'
    )

    for category in df_final['Category'].dropna().unique():
        cat_df = df_final[df_final['Category'] == category].copy()
        file_name = f"{platform}_CloseReading_{category}.csv"
        cat_df.to_csv(
            os.path.join(SPACE_CLOSE_DIR, file_name),
            index=False,
            encoding='utf-8-sig'
        )
        print("saved:", file_name)

for p in ["ptt", "theqoo", "reddit"]:
    pull_space_close_reading(p)

# color groups

In [ ]:
COLOR_GROUPS = {
    "ptt": {
        "White": ['白衣', '白色','全白','襲白','穿白','褲白鞋',
                  '白鞋','白點','白袍','白光','白線','青白','發白','白面','白霧',
                  '白布','白影'],
        "Red": [ '紅紙','紅裙','紅影','紅點','紅絲','染紅','紅字畫','紅布','紅褲',
                '發紅','紅線','紅光','血色'],
        "Black": ['黑影', '黑鬼','黑氣','黑色','黑青','烏黑','黑衣服','黑服',
                  '全黑','焦黑','黑衣','黑人','發黑','灰黑色','黑霧','黑煙','暗黑',
                  '黑面','黑衣人'],
        "Green": ['綠氣','綠霧','綠衣','綠光','綠綠','陣綠光','草綠服',
                  '變綠','青色'],
        "Blue": ['藍光','藍面','藏青色','鬼火'],
        "Yellow": ['黃衣','黃光'],
        "Pink": ['粉色'],
        "Brown": ['褐色','咖啡色'],
        "Grey": ['灰色','灰白色'],
        "Orange": ['橘色','金色'],
        "Purple": ['紫衣']
    },
    "theqoo": {
        "White": ['창백', '소복','흰색','하얀색','백발'],
        "Red": ['빨간색','피눈물'],
        "Black": ['흑색', '그림자',],
        "Blue": ['파란색','도깨비불','파란색','파랑색'],
        "Green": ['초록'],
        "Brown": ['갈색'],
        "Pink": ['핑크색'],
        "Grey": ['회백색','회색'],
        "Purple": ['연보라색']
    },
    "reddit": {
        "White": ['pale','whiteish','white','colorless'],
        "Red": ['redheade','reddish','red','bloody','bloodstained',
                'bloodied','blood'],
        "Pink": ['pink',],
        "Black": ['darken','dark','darkest','darker','darkness','blacken',
                  'blacker','blackness','black','pitch','shadow person','shadow figure',
                  'shadows','shadow','shadowy','shadowed'],
        "Grey": ['grey', 'gray','greyish','grayish'],
        "Green": ['green','greenish'],
        "Blue": ['blue','blueish'],
        "Purple": ['purple'],
        "Brown": ['brown','brownish']
    }
}

# ptt/color

In [ ]:
summary_ptt_color, keyword_ptt_color, details_ptt_color = run_domain_analysis_from_all(
    input_file=ALL_MENTIONS_PATH,
    output_dir=COLOR_OUTPUT_DIR,
    platform_name="ptt",
    groups_dict=COLOR_GROUPS["ptt"],
    output_prefix="color",
    scan_col_preference='context',
    norm_per=1000
)

# theqoo/color

In [ ]:
summary_theqoo_color, keyword_theqoo_color, details_theqoo_color = run_domain_analysis_from_all(
    input_file=ALL_MENTIONS_PATH,
    output_dir=COLOR_OUTPUT_DIR,
    platform_name="theqoo",
    groups_dict=COLOR_GROUPS["theqoo"],
    output_prefix="color",
    scan_col_preference='context',
    norm_per=1000
)

# reddit/color

In [ ]:
summary_reddit_color, keyword_reddit_color, details_reddit_color = run_domain_analysis_from_all(
    input_file=ALL_MENTIONS_PATH,
    output_dir=COLOR_OUTPUT_DIR,
    platform_name="reddit",
    groups_dict=COLOR_GROUPS["reddit"],
    output_prefix="color",
    scan_col_preference='context',
    norm_per=1000
)

# close reading pullout（color）

In [ ]:
COLOR_DETAIL_PATHS = {
    "ptt": os.path.join(COLOR_OUTPUT_DIR, "ptt_color_matched_sentences_details.csv"),
    "theqoo": os.path.join(COLOR_OUTPUT_DIR, "theqoo_color_matched_sentences_details.csv"),
    "reddit": os.path.join(COLOR_OUTPUT_DIR, "reddit_color_matched_sentences_details.csv")
}

COLOR_CLOSE_DIR = os.path.join(CLOSE_READING_DIR, "Color_Analysis")
ensure_dir(COLOR_CLOSE_DIR)

def pull_color_close_reading(platform):
    detail_path = COLOR_DETAIL_PATHS[platform]
    if not os.path.exists(detail_path):
        print("missing:", detail_path)
        return

    df_details = pd.read_csv(detail_path)
    df_raw = load_raw_articles(platform)

    merge_key = 'article_id' if 'article_id' in df_details.columns else 'Story_ID'

    # 兼容 raw 檔不同全文欄位名稱
    raw_cols = ['article_id', 'Story_ID']
    if 'content' in df_raw.columns:
        raw_cols.append('content')
    elif 'text' in df_raw.columns:
        raw_cols.append('text')
    else:
        raise ValueError(f"No content/text column found in raw file for {platform}")

    df_raw_sub = df_raw[raw_cols].copy()
    df_raw_sub = df_raw_sub.rename(columns={
        'content': 'full_text',
        'text': 'full_text'
    })

    df_final = pd.merge(
        df_details,
        df_raw_sub,
        on=merge_key,
        how='left'
    )

    cat_column = 'Category' if 'Category' in df_final.columns else 'Color'

    for cat in df_final[cat_column].dropna().unique():
        cat_df = df_final[df_final[cat_column] == cat].copy()
        file_name = f"{platform}_ColorReading_{cat}.csv"
        cat_df.to_csv(
            os.path.join(COLOR_CLOSE_DIR, file_name),
            index=False,
            encoding='utf-8-sig'
        )
        print("saved:", file_name)

for p in ["ptt", "theqoo", "reddit"]:
    pull_color_close_reading(p)

# haunting methods groups

In [ ]:
HAUNTING_METHOD_GROUPS = {
    "ptt": {
        "Sound": [ '敲打', '尖叫', '嚎啕大哭', '哭泣','叫喚','叫出','狂叫','大叫','大吼大叫',
                  '吠叫','喊叫','哀叫','哭叫','叫住','叫著','嚎叫','呼叫','叫喊','怪叫','吼叫',
                   '狗叫','叫醒','敲門','敲著','敲敲','敲擊','鬼哭','痛哭','哭醒','哭喊','大哭',
                   '爆哭','嘶吼','吼聲','狂吼','怒吼','斯吼','人聲','女聲','低聲','小聲','作聲','輕聲',
                   '大聲','門聲','鍵盤聲','聲音','聲調','高聲','出聲','聲響','彈珠聲','氣聲','鬼聲','鈴聲',
                   '男聲','雜聲','鈴鐺聲','銅鈴聲','翁翁聲','回音','童音','噪音','雜音','氣音','吹口哨','風聲',
                   '唱歌','歌聲','獰笑','發笑','咧嘴一笑','嘻笑','笑容','笑笑','笑著','狂笑','微笑',
                   '奸笑','大笑','冷笑'],
        "Touch": ['抓著','抓出去','抓走','抓痕','抓住','抓到','拉著','拉下去','拉腳','往下拉','猛拍',
                  '拍肩膀','拍打','觸碰','碰觸','碰到','觸摸','撫摸','手摸','摸到','摸摸','摸著','山風','海風',
                  '冷風','狂風','大風','微風','寒風','涼風','推開','推下去','推著','推下'],
        "Visual_Appearance": ['注視','出現', '站著','站立','看見','盯著','盯上','對視'],
        "Movement": ['消失','進房','靠窗','跟到','跟來','跟去','跟壓','壓床','壓著',
                     '上樓','下樓','墜樓','跳樓','上頂樓','跳海','落海','打開',
                     '壓制','鬼壓','壓到','被壓','壓住','人壓','壓過','擠壓',
                     '飄出','飄去','飄移','飄起','飄移','飄過','飄走','飄動','飄浮','飄著',
                     '飄進','穿越','穿門','穿梭','狂奔','奔跑','跑步','跑掉','飛奔',
                     '附著','依附','附身'],
    },
    "theqoo": {
        "Sound": ['휘파람','물소리','노랫소리','소리','목소리','발소리','소리치',
                  '울컥','울리','울음','외치','웃음'],
        "Touch": ['손길','바람','붙잡히','잡히','만지','어루만지','만지작','끌어안','끌려가',
                  '밀어내'],
        "Visual_Appearance": [ '바라보','돌아보','노려보','지켜보','내려다보','쳐다보','보이다',
                               '나타나','다가서','뒤돌아서'],
        "Movement": ['가위눌리','가위눌림','가위','따라오','누르','따라가','지나가','지나','지나치','사라지',
                     '움직이','뛰쳐나오','뛰어오','뛰어나가','씌이','빙의'],
    },
    "reddit": {
        "Sound": ['scream','screaming','voice','disembodied voice','disembodied','shrieking',
                   'shriek','crying','cry','breath','breathing','breathe','laughing','laugher',
                  'laughter','laugh','whistle','sound','sounding','shout','shouting','shouted'],
        "Touch": ['pull','tapping','tap','grab','touch','touching','push','slam','pushing','slamming'],
        "Visual_Appearance": [ 'watching','reappear','appear',
                               'standing','stare','watch','see','unseen','saw'],
        "Movement": ['move','disappear','disappearing','dissappeare','disappearance','following','follow',
                     'walking','walk','vanish','vanishing','float','floated','running','run',
                     'possessed','possess','possession'],
    }
}

# ptt/haunting methods

In [ ]:
summary_ptt_method, keyword_ptt_method, details_ptt_method = run_domain_analysis_from_all(
    input_file=ALL_MENTIONS_PATH,
    output_dir=METHOD_OUTPUT_DIR,
    platform_name="ptt",
    groups_dict=HAUNTING_METHOD_GROUPS["ptt"],
    output_prefix="haunting_method",
    scan_col_preference='context',
    norm_per=1000
)

# theqoo/haunting methods

In [ ]:
summary_theqoo_method, keyword_theqoo_method, details_theqoo_method = run_domain_analysis_from_all(
    input_file=ALL_MENTIONS_PATH,
    output_dir=METHOD_OUTPUT_DIR,
    platform_name="theqoo",
    groups_dict=HAUNTING_METHOD_GROUPS["theqoo"],
    output_prefix="haunting_method",
    scan_col_preference='context',
    norm_per=1000
)

# reddit/haunting methods

In [ ]:
summary_reddit_method, keyword_reddit_method, details_reddit_method = run_domain_analysis_from_all(
    input_file=ALL_MENTIONS_PATH,
    output_dir=METHOD_OUTPUT_DIR,
    platform_name="reddit",
    groups_dict=HAUNTING_METHOD_GROUPS["reddit"],
    output_prefix="haunting_method",
    scan_col_preference='context',
    norm_per=1000
)

# close reading pullout (haunting methods)

In [ ]:
METHOD_DETAIL_PATHS = {
    "ptt": os.path.join(METHOD_OUTPUT_DIR, "ptt_haunting_method_matched_sentences_details.csv"),
    "theqoo": os.path.join(METHOD_OUTPUT_DIR, "theqoo_haunting_method_matched_sentences_details.csv"),
    "reddit": os.path.join(METHOD_OUTPUT_DIR, "reddit_haunting_method_matched_sentences_details.csv")
}

METHOD_CLOSE_DIR = os.path.join(CLOSE_READING_DIR, "Haunting_methods_Analysis")
ensure_dir(METHOD_CLOSE_DIR)

def pull_methods_close_reading(platform):
    detail_path = METHOD_DETAIL_PATHS[platform]
    if not os.path.exists(detail_path):
        print("missing:", detail_path)
        return

    df_details = pd.read_csv(detail_path)
    df_raw = load_raw_articles(platform)

    merge_key = 'article_id' if 'article_id' in df_details.columns else 'Story_ID'

    # 兼容 raw 檔不同全文欄位名稱
    raw_cols = ['article_id', 'Story_ID']
    if 'content' in df_raw.columns:
        raw_cols.append('content')
    elif 'text' in df_raw.columns:
        raw_cols.append('text')
    else:
        raise ValueError(f"No content/text column found in raw file for {platform}")

    df_raw_sub = df_raw[raw_cols].copy()
    df_raw_sub = df_raw_sub.rename(columns={
        'content': 'full_text',
        'text': 'full_text'
    })

    df_final = pd.merge(
        df_details,
        df_raw_sub,
        on=merge_key,
        how='left'
    )

    cat_column = 'Category' if 'Category' in df_final.columns else 'Haunting methods'

    for cat in df_final[cat_column].dropna().unique():
        cat_df = df_final[df_final[cat_column] == cat].copy()
        file_name = f"{platform}_HauntingMethodsReading_{cat}.csv"
        cat_df.to_csv(
            os.path.join(METHOD_CLOSE_DIR, file_name),
            index=False,
            encoding='utf-8-sig'
        )
        print("saved:", file_name)

for p in ["ptt", "theqoo", "reddit"]:
    pull_methods_close_reading(p)

# article-level robust checked

In [ ]:
def run_article_level_presence(
    input_file,
    output_dir,
    platform_name,
    groups_dict,
    output_prefix,
    scan_col_preference='context',
    norm_per=100
):
    ensure_dir(output_dir)

    df_ent = pd.read_csv(input_file)
    df_ent = standardize_mentions_df(df_ent)

    # 一定要先按平台篩選
    df_ent['platform'] = df_ent['platform'].astype(str).str.lower().str.strip()
    df_ent = df_ent[df_ent['platform'] == platform_name.lower()].copy()

    df_ghosts = prepare_ghost_mentions(df_ent)

    platform_lang = get_lang_key(df_ghosts['lang'].dropna().iloc[0]) if len(df_ghosts) else 'en'

    base_articles = (
        df_ghosts[['article_id', 'gender_new']]
        .drop_duplicates()
        .groupby('gender_new')
        .size()
        .to_dict()
    )

    rows = []

    for category, keywords in groups_dict.items():
        pattern, _ = compile_keyword_pattern(keywords, lang=platform_lang)

        hit_rows = []
        for _, row in df_ghosts.iterrows():
            scan_text = pick_text_for_scan(row) if scan_col_preference == 'context' else str(row['sentence'])
            found = find_matches(scan_text, pattern, lang=row['lang'])
            if found:
                hit_rows.append({
                    'article_id': row['article_id'],
                    'gender_new': row['gender_new'],
                    'Category': category
                })

        if len(hit_rows) == 0:
            continue

        hit_df = pd.DataFrame(hit_rows).drop_duplicates()

        counts = (
            hit_df.groupby(['Category', 'gender_new'])['article_id']
            .nunique()
            .reset_index(name='article_hits')
        )

        for _, r in counts.iterrows():
            gender = r['gender_new']
            denom = base_articles.get(gender, 0)
            rows.append({
                'Platform': platform_name,
                'Category': r['Category'],
                'Gender': gender,
                'Article_Hits': r['article_hits'],
                'Base_Articles': denom,
                'Article_Level_Norm': (r['article_hits'] / denom * norm_per) if denom > 0 else 0
})

    out_df = pd.DataFrame(rows)
    out_path = os.path.join(output_dir, f"{platform_name}_{output_prefix}_article_level_presence.csv")
    out_df.to_csv(out_path, index=False, encoding='utf-8-sig')
    print("saved:", out_path)

    # ── Article-level statistical test by category ─────────────────────
    stat_rows = []

    if len(out_df) > 0:
        for category in out_df['Category'].unique():
            sub = out_df[out_df['Category'] == category].copy()

            female_hits = sub.loc[sub['Gender'] == 'Female', 'Article_Hits'].sum() if 'Female' in sub['Gender'].values else 0
            male_hits   = sub.loc[sub['Gender'] == 'Male', 'Article_Hits'].sum() if 'Male' in sub['Gender'].values else 0

            female_base = sub.loc[sub['Gender'] == 'Female', 'Base_Articles'].max() if 'Female' in sub['Gender'].values else base_articles.get('Female', 0)
            male_base   = sub.loc[sub['Gender'] == 'Male', 'Base_Articles'].max() if 'Male' in sub['Gender'].values else base_articles.get('Male', 0)

            female_nohit = female_base - female_hits
            male_nohit   = male_base - male_hits

            table = [
                [female_hits, female_nohit],
                [male_hits, male_nohit]
            ]

            # 預設值
            test_used = None
            chi2_stat = np.nan
            p_value = np.nan
            dof = np.nan
            expected_min = np.nan
            odds_ratio = np.nan

            try:
                chi2, p, dof_val, expected = chi2_contingency(table)
                expected_min = expected.min()
                chi2_stat = chi2
                dof = dof_val

                # 若期望次數太小，改用 Fisher
                if expected_min < 5:
                    odds_ratio, p_fisher = fisher_exact(table)
                    p_value = p_fisher
                    test_used = 'Fisher'
                else:
                    p_value = p
                    test_used = 'Chi-square'

            except Exception:
                # fallback
                try:
                    odds_ratio, p_fisher = fisher_exact(table)
                    p_value = p_fisher
                    test_used = 'Fisher'
                except Exception:
                    p_value = np.nan
                    test_used = 'Failed'

            # significance stars
            if pd.isna(p_value):
                sig = ''
            elif p_value < 0.001:
                sig = '***'
            elif p_value < 0.01:
                sig = '**'
            elif p_value < 0.05:
                sig = '*'
            else:
                sig = ''

            stat_rows.append({
                'Platform': platform_name,
                'Category': category,
                'Female_Hits': female_hits,
                'Female_NoHit': female_nohit,
                'Female_Base_Articles': female_base,
                'Male_Hits': male_hits,
                'Male_NoHit': male_nohit,
                'Male_Base_Articles': male_base,
                'Female_Pct': (female_hits / female_base * 100) if female_base > 0 else 0,
                'Male_Pct': (male_hits / male_base * 100) if male_base > 0 else 0,
                'Test_Used': test_used,
                'Chi2_Stat': chi2_stat,
                'DOF': dof,
                'Expected_Min': expected_min,
                'Odds_Ratio': odds_ratio,
                'P_Value': p_value,
                'Significance': sig
            })

    stat_df = pd.DataFrame(stat_rows)
    stat_path = os.path.join(output_dir, f"{platform_name}_{output_prefix}_article_level_stats.csv")
    stat_df.to_csv(stat_path, index=False, encoding='utf-8-sig')
    print("saved:", stat_path)

    if len(out_df) > 0:
      pct_plot = (
        out_df.pivot(index='Category', columns='Gender', values='Article_Level_Norm')
        .fillna(0)
        .reset_index()
    )

      if 'Female' not in pct_plot.columns:
          pct_plot['Female'] = 0
      if 'Male' not in pct_plot.columns:
          pct_plot['Male'] = 0

      pct_plot['Total_Norm'] = pct_plot['Female'] + pct_plot['Male']
      pct_plot = pct_plot.sort_values('Total_Norm', ascending=True)

      y = np.arange(len(pct_plot))
      h = 0.35

      plt.figure(figsize=(10, max(4, len(pct_plot) * 0.6)))
      plt.barh(y + h/2, pct_plot['Female'], h, label='Female', color='#f0a1a8')
      plt.barh(y - h/2, pct_plot['Male'], h, label='Male', color='#a1c4f0')
      plt.yticks(y, pct_plot['Category'])
      plt.xlabel(f"Normalized Article Frequency (per {norm_per} articles)")
      plt.title(f"{platform_name.upper()} {output_prefix.upper()} Article-Level Category Distribution by Gender (Normalized)")
      plt.legend()
      plt.tight_layout()
      plt.savefig(
        os.path.join(output_dir, f"{platform_name}_{output_prefix}_article_level_norm_bar.png"),
        dpi=300
    )
      plt.show()



    if len(out_df) > 0:
      raw_plot = (
        out_df.pivot(index='Category', columns='Gender', values='Article_Hits')
        .fillna(0)
        .reset_index()
    )

      if 'Female' not in raw_plot.columns:
          raw_plot['Female'] = 0
      if 'Male' not in raw_plot.columns:
          raw_plot['Male'] = 0

      raw_plot['Total_Hits'] = raw_plot['Female'] + raw_plot['Male']
      raw_plot = raw_plot.sort_values('Total_Hits', ascending=True)

      y = np.arange(len(raw_plot))
      h = 0.35

      plt.figure(figsize=(10, max(4, len(raw_plot) * 0.6)))
      plt.barh(y + h/2, raw_plot['Female'], h, label='Female', color='#f0a1a8')
      plt.barh(y - h/2, raw_plot['Male'], h, label='Male', color='#a1c4f0')
      plt.yticks(y, raw_plot['Category'])
      plt.xlabel("Article Hits (Number of Articles)")
      plt.title(f"{platform_name.upper()} {output_prefix.upper()} Article-Level Category Distribution by Gender (Raw Count)")
      plt.legend()
      plt.tight_layout()
      plt.savefig(
        os.path.join(output_dir, f"{platform_name}_{output_prefix}_article_level_raw_bar.png"),
        dpi=300
    )
      plt.show()

    return out_df

In [ ]:
# place
_ = run_article_level_presence(
    input_file=ALL_MENTIONS_PATH,
    output_dir=SPACE_OUTPUT_DIR,
    platform_name="ptt",
    groups_dict=SPACE_GROUPS["ptt"],
    output_prefix="space"
)

_ = run_article_level_presence(
    input_file=ALL_MENTIONS_PATH,
    output_dir=SPACE_OUTPUT_DIR,
    platform_name="theqoo",
    groups_dict=SPACE_GROUPS["theqoo"],
    output_prefix="space"
)

_ = run_article_level_presence(
    input_file=ALL_MENTIONS_PATH,
    output_dir=SPACE_OUTPUT_DIR,
    platform_name="reddit",
    groups_dict=SPACE_GROUPS["reddit"],
    output_prefix="space"
)

# color
_ = run_article_level_presence(
    input_file=ALL_MENTIONS_PATH,
    output_dir=COLOR_OUTPUT_DIR,
    platform_name="ptt",
    groups_dict=COLOR_GROUPS["ptt"],
    output_prefix="color"
)

_ = run_article_level_presence(
    input_file=ALL_MENTIONS_PATH,
    output_dir=COLOR_OUTPUT_DIR,
    platform_name="theqoo",
    groups_dict=COLOR_GROUPS["theqoo"],
    output_prefix="color"
)

_ = run_article_level_presence(
    input_file=ALL_MENTIONS_PATH,
    output_dir=COLOR_OUTPUT_DIR,
    platform_name="reddit",
    groups_dict=COLOR_GROUPS["reddit"],
    output_prefix="color"
)

# haunting methods
_ = run_article_level_presence(
    input_file=ALL_MENTIONS_PATH,
    output_dir=METHOD_OUTPUT_DIR,
    platform_name="ptt",
    groups_dict=HAUNTING_METHOD_GROUPS["ptt"],
    output_prefix="haunting_method"
)

_ = run_article_level_presence(
    input_file=ALL_MENTIONS_PATH,
    output_dir=METHOD_OUTPUT_DIR,
    platform_name="theqoo",
    groups_dict=HAUNTING_METHOD_GROUPS["theqoo"],
    output_prefix="haunting_method"
)
_ = run_article_level_presence(
    input_file=ALL_MENTIONS_PATH,
    output_dir=METHOD_OUTPUT_DIR,
    platform_name="reddit",
    groups_dict=HAUNTING_METHOD_GROUPS["reddit"],
    output_prefix="haunting_method"
)